# HuggingFace Transformers: API, Models and Fine-Tuning Techniques

---

### Using the ```pipeline()``` from the ```transformers``` library

> NOTE: transformers 5.x has introduced breaking changes in their architecture.

> Thus, many tasks like question-answering / summarization are not available via
> the pipeline() API.  

> Also, many models that are not updated recently would fail to work with transformers 5.x. This includes a lot of ASR, Text-To-Speech and Image generation models.

> The fix is currently underway. Until then, it is better to explore transformers 4.x for our examples where transformers 5.x breaks.

To create a seperate environment for experimenting with transformers 4.x:
```bash
   conda create -n hf4_env python=3.11 jupyter -y
   conda activate hf4_env
   conda install -c conda-forge "transformers>=4.40,<5.0" \
                    datasets evaluate accelerate          \
                    pytorch torchvision torchaudio -y
```

> NOTE: If you get SSLVerificationError, try running the following command:
```bash
conda config --set ssl_verify false
```

And then try installing

In [11]:
import transformers
import torch
import sys
print("Transformers version:", transformers.__version__)
print("Torch version:", torch.__version__)
print("Python version:", sys.version)

print("Accelerator: ", torch.accelerator.current_accelerator())

Transformers version: 4.57.6
Torch version: 2.10.0
Python version: 3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]
Accelerator:  mps


In [12]:
from transformers import pipeline

pipeline?

Signature:
pipeline(
    task: Optional[str] = None,
    model: Union[str, ForwardRef('PreTrainedModel'), ForwardRef('TFPreTrainedModel'), NoneType] = None,
    config: Union[str, transformers.configuration_utils.PretrainedConfig, NoneType] = None,
    tokenizer: Union[str, transformers.tokenization_utils.PreTrainedTokenizer, ForwardRef('PreTrainedTokenizerFast'), NoneType] = None,
    feature_extractor: Union[ForwardRef('SequenceFeatureExtractor'), str, NoneType] = None,
    image_processor: Union[str, transformers.image_processing_utils.BaseImageProcessor, NoneType] = None,
    processor: Union[str, transformers.processing_utils.ProcessorMixin, NoneType] = None,
    framework: Optional[str] = None,
    revision: Optional[str] = None,
    use_fast: bool = True,
    token: Union[str, bool, NoneType] = None,
    device: Union[int, str, ForwardRef('torch.device'), NoneType] = None,
    device_map: Union[str, dict[str, Union[int, str]], NoneType] = None,
    dtype: Union[str, ForwardRef('

In [14]:
from transformers import pipeline
classifier = pipeline(task="sentiment-analysis", 
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
result = classifier("I love machine learning!")
print(result)

result = classifier("Transformers are amazing but complex.")
print(result)

result = classifier("I hate bugs in my code.")
print(result)
print(result[0]["label"], result[0]["score"])

Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9998431205749512}]
[{'label': 'POSITIVE', 'score': 0.9989979863166809}]
[{'label': 'NEGATIVE', 'score': 0.999018669128418}]
NEGATIVE 0.999018669128418


---

### Pipeline Examples: NLP Tasks


In [15]:
# 1. Sentiment Analysis
classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
results = classifier(["Great product!", "Terrible experience..."])
# [{'label': 'POSITIVE', 'score': 0.99}, {'label': 'NEGATIVE', 'score': 0.98}]
print(results)


Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9998729228973389}, {'label': 'NEGATIVE', 'score': 0.9996969699859619}]


In [ ]:
# Silencing warnings for cleaner output (needed for transformers 5.x)
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)

In [16]:
# 2. Named Entity Recognition (ner)
from transformers import pipeline
ner = pipeline(task="ner", 
               aggregation_strategy="simple", 
               model="dbmdz/bert-large-cased-finetuned-conll03-english")

#text = "Elon Musk founded SpaceX in Hawthorne, California."
text = "Chandrashekar is a managing director of Slashprog Technologies, lives in Chennai, India and trains for Qualcomm."
entities = ner(text)


Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [17]:
print(entities)


[{'entity_group': 'PER', 'score': np.float32(0.9917142), 'word': 'Chandrashekar', 'start': 0, 'end': 13}, {'entity_group': 'ORG', 'score': np.float32(0.9913398), 'word': 'Slashprog Technologies', 'start': 40, 'end': 62}, {'entity_group': 'LOC', 'score': np.float32(0.9992513), 'word': 'Chennai', 'start': 73, 'end': 80}, {'entity_group': 'LOC', 'score': np.float32(0.99959594), 'word': 'India', 'start': 82, 'end': 87}, {'entity_group': 'ORG', 'score': np.float32(0.974483), 'word': 'Qualcomm', 'start': 103, 'end': 111}]


In [18]:

for entity in entities:
    print(f"Entity: {entity['word']}, Type: {entity['entity_group']}, Score: {entity['score']:.4f}") 

Entity: Chandrashekar, Type: PER, Score: 0.9917
Entity: Slashprog Technologies, Type: ORG, Score: 0.9913
Entity: Chennai, Type: LOC, Score: 0.9993
Entity: India, Type: LOC, Score: 0.9996
Entity: Qualcomm, Type: ORG, Score: 0.9745


In [4]:
p = pipeline("question-answering", model="distilbert/distilbert-base-cased-distilled-squad")
p

Device set to use mps:0


In [ ]:
pipeline?

Signature:
pipeline(
    task: 'str | None' = None,
    model: 'str | PreTrainedModel | None' = None,
    config: 'str | PreTrainedConfig | None' = None,
    tokenizer: 'str | PreTrainedTokenizer | PreTrainedTokenizerFast | None' = None,
    feature_extractor: 'str | FeatureExtractionMixin | None' = None,
    image_processor: 'str | BaseImageProcessor | None' = None,
    video_processor: 'str | BaseVideoProcessor | None' = None,
    processor: 'str | ProcessorMixin | None' = None,
    revision: 'str | None' = None,
    use_fast: 'bool' = True,
    token: 'str | bool | None' = None,
    device: 'int | str | torch.device | None' = None,
    device_map: 'str | dict[str, int | str] | None' = None,
    dtype: 'str | torch.dtype | None' = 'auto',
    trust_remote_code: 'bool | None' = None,
    model_kwargs: 'dict[str, Any] | None' = None,
    pipeline_class: 'Any | None' = None,
    **kwargs: 'Any',
) -> 'Pipeline'
Docstring:
Utility factory method to build a [`Pipeline`].

A pipeline consi

In [ ]:
from transformers import pipeline

text_gen = pipeline("text-generation", model="gpt2")
result = text_gen("The distance between Earth and Moon is")
print(result[0]["generated_text"])

Device set to use mps:0
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The distance between Earth and Moon is 12.6 light years, or about 26.8 astronomical units (AU), depending on the moon's orbital position, and is about one-third the diameter of Earth and about one-twentieth the diameter of Jupiter.

In the early part of the 20th century, the moon's orbit around Earth was around 65,000 kilometers (50,000 miles) from Earth.

In 1998, the European Space Agency's Rosetta spacecraft passed within the same distance in its mission's first orbit. The spacecraft had returned images of the surface of the moon's surface, but it was too far away to receive data on the moon's surface.

And by 2009, Rosetta had returned images of the moon's surface after just over three years, and some of the data was lost to the comet.

"The information that we have about the surface of the moon in this study was lost to the comet and its companion comet, so that's a real shame," said Robert H. Lehmiller, a geoscientist at NASA's Jet Propulsion Laboratory in Pasadena, California, w

In [21]:
from huggingface_hub import list_models

qa_models = list_models(filter="summarization", sort="downloads", limit=5)
for model in qa_models:
    print(f"Model: {model.modelId}, Downloads: {model.downloads:,}")

Model: google-t5/t5-small, Downloads: 26,349,824
Model: google-t5/t5-base, Downloads: 1,520,527
Model: facebook/bart-large-cnn, Downloads: 1,408,100
Model: sshleifer/distilbart-cnn-12-6, Downloads: 754,640
Model: google-t5/t5-3b, Downloads: 255,480


In [22]:
# 3. Text-Generation

text_gen = pipeline("text-generation", model='openai-community/gpt2')

result = text_gen(
    "What is the distance between the Earth and the Moon in kilometers?"
)

#print(result)
print(result[0]['generated_text'])


Device set to use mps:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is the distance between the Earth and the Moon in kilometers?

The distance between the Earth and the Moon in kilometers is exactly the distance between the Earth and the Moon, which is 1,500 kilometers in the satellite track. That's the distance between the Earth and the Moon, which is 1,500 kilometers in the satellite track.

But if you looked at the satellites, they are all orbiting in the same orbit. And if you look at the satellites, they are all orbiting in the same orbit. And if you look at the satellites, you can tell that the Earth is not just in the same orbit as the moon. The Moon is not only in the same orbit as the Earth, but it's also in a different orbit and it's a different satellite.

So there is a lot of difference between the Earth and the Moon. If that's the case, we might have a large number of satellites circling the Moon with the same speed and with the same orbit.

Are there any other satellites orbiting the planet that you can do something about and what d

---

### Using the low-level API for more tasks

In [23]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model_inputs = tokenizer("This distance between the Earth and the Moon in kilometers is", return_tensors="pt")
generated_ids = model.generate(**model_inputs, max_new_tokens=20)
generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(generated_text)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


This distance between the Earth and the Moon in kilometers is about 1.5 times the distance between the Earth and the Moon in miles.

The distance


In [26]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openai-community/gpt2"
#model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer


GPT2TokenizerFast(name_or_path='openai-community/gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [27]:

model = AutoModelForCausalLM.from_pretrained(model_name)
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [30]:

text = "This distance between the Earth and the Moon in kilometers is"
model_inputs = tokenizer(text, return_tensors="pt")

model_inputs


{'input_ids': tensor([[ 1212,  5253,  1022,   262,  3668,   290,   262,  6869,   287, 18212,
           318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [36]:

generated_ids = model.generate(**model_inputs, max_new_tokens=50)
generated_ids.squeeze()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tensor([ 1212,  5253,  1022,   262,  3668,   290,   262,  6869,   287, 18212,
          318,   546,   352,    13,    20,  1661,   262,  5253,  1022,   262,
         3668,   290,   262,  6869,   287,  4608,    13,   198,   198,   464,
         5253,  1022,   262,  3668,   290,   262,  6869,   287, 18212,   318,
          546,   352,    13,    20,  1661,   262,  5253,  1022,   262,  3668,
          290,   262,  6869,   287,  4608,    13,   383,  5253,  1022,   262,
         3668])

In [37]:
generated_text = tokenizer.decode(generated_ids.squeeze())
print(generated_text)


This distance between the Earth and the Moon in kilometers is about 1.5 times the distance between the Earth and the Moon in miles.

The distance between the Earth and the Moon in kilometers is about 1.5 times the distance between the Earth and the Moon in miles. The distance between the Earth


---

### Question-Answering without ```pipeline()``` API

In [59]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased-distilled-squad")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")

question = "How many parameters does BLOOM support?"
context = "BLOOM has 176 billion parameters and can generate text in 46 languages natural languages and 13 programming languages."

inputs = tokenizer(question, context, return_tensors="pt")
inputs


{'input_ids': tensor([[  101,  2129,  2116, 11709,  2515, 13426,  2490,  1029,   102, 13426,
          2038, 18561,  4551, 11709,  1998,  2064,  9699,  3793,  1999,  4805,
          4155,  3019,  4155,  1998,  2410,  4730,  4155,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1]])}

In [40]:

outputs = model(**inputs)
outputs


QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[-4.7285e+00, -4.9716e+00, -5.7424e+00, -5.8973e+00, -6.0992e+00,
         -6.7129e+00, -5.9491e+00, -6.7422e+00, -7.1948e+00, -5.2413e+00,
         -6.4316e-03, -3.2589e+00,  1.8859e+00, -2.9998e+00, -3.3188e+00,
         -4.4051e+00, -2.8847e+00, -3.3174e+00, -2.0914e+00, -3.6406e+00,
          2.1037e+00, -3.2412e+00, -2.5054e+00, -4.1058e+00, -4.2817e+00,
          7.6575e+00, -3.1638e+00, -3.5964e+00, -4.5163e+00, -5.2413e+00]],
       grad_fn=<CloneBackward0>), end_logits=tensor([[-1.3434, -5.4354, -5.4226, -6.0483, -5.0513, -6.6298, -5.5360, -6.2843,
         -6.4340, -1.1506, -2.1572, -5.3636, -0.4894,  0.8032, -0.5443, -4.2107,
         -5.5277, -4.8551, -2.3976, -4.8884,  2.1505, -0.3825, -2.6784,  0.4288,
         -4.6941,  7.2952, -0.1557,  4.4586,  1.8896, -1.1505]],
       grad_fn=<CloneBackward0>), hidden_states=None, attentions=None)

In [73]:
s = outputs.start_logits.argsort(descending=True).squeeze()[1]
e = outputs.end_logits.argsort(descending=True).squeeze()[1]
print(s, e)
inputs["input_ids"].squeeze()[s:e + 1]
t = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze()[s:e + 1])
t

tensor(20) tensor(27)


['languages',
 'natural',
 'languages',
 'and',
 '13',
 'programming',
 'languages',
 '.']

In [74]:
answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()

In [56]:
i = inputs["input_ids"].squeeze()[answer_start_index:answer_end_index + 1]
i

tensor([4730])

In [57]:
t = tokenizer.convert_ids_to_tokens(i)
t

['programming']

In [58]:
tokenizer.convert_tokens_to_string(t)

'programming'

In [75]:

answer_start_index = outputs.start_logits.argmax()
answer_end_index = outputs.end_logits.argmax()
answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start_index:answer_end_index+1]))
print(answer)


programming


In [38]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased-distilled-squad")
tokenizer?

Signature:     
tokenizer(
    text: Union[str, list[str], list[list[str]], NoneType] = None,
    text_pair: Union[str, list[str], list[list[str]], NoneType] = None,
    text_target: Union[str, list[str], list[list[str]], NoneType] = None,
    text_pair_target: Union[str, list[str], list[list[str]], NoneType] = None,
    add_special_tokens: bool = True,
    padding: Union[bool, str, transformers.utils.generic.PaddingStrategy] = False,
    truncation: Union[bool, str, transformers.tokenization_utils_base.TruncationStrategy, NoneType] = None,
    max_length: Optional[int] = None,
    stride: int = 0,
    is_split_into_words: bool = False,
    pad_to_multiple_of: Optional[int] = None,
    padding_side: Optional[str] = None,
    return_tensors: Union[str, transformers.utils.generic.TensorType, NoneType] = None,
    return_token_type_ids: Optional[bool] = None,
    return_attention_mask: Optional[bool] = None,
    return_overflowing_tokens: bool = False,
    return_special_tokens_mask: bool

---

### Question-Answering using the ```pipeline()``` API

In [ ]:
from transformers import pipeline

#model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")


context = """The Linux kernel project is a free and open-source, monolithic, Unix-like operating system kernel. 
The Linux kernel was conceived and created in 1991 by Linus Torvalds for his personal computer, and it has since 
grown to support a wide variety of computer architectures, including the x86, ARM, and RISC-V instruction sets. 
The Linux kernel is released under the GNU General Public License version 2 (GPLv2), which allows anyone to view, 
modify, and distribute the source code. The kernel is developed by a large community of developers from around 
the world, with contributions from individuals, companies, and organizations. It is used as the foundation for 
many operating systems, including popular distributions such as Ubuntu, Fedora, and Debian.
"""

#question = "Who created the Linux kernel?"
#question = "When was the Linux kernel created?"
#question = "What is the license of the Linux kernel?"
#question = "What is the Linux kernel used for?"
#question = "What is the Linux kernel?"
question = "Which are popular distributions that use the Linux kernel?"

qa_pipeline = pipeline(task="question-answering", 
                       model="distilbert-base-cased-distilled-squad", 
                       tokenizer="distilbert-base-cased-distilled-squad")
answer = qa_pipeline(question=question, context=context)
print(answer["answer"])
print(answer)


Device set to use mps:0


operating system kernel
{'score': 0.009361447766423225, 'start': 74, 'end': 97, 'answer': 'operating system kernel'}


In [79]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

#model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-cased-distilled-squad")
#tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased-distilled-squad")

qa_pipeline = pipeline("question-answering", 
                       model="distilbert-base-uncased-distilled-squad", 
                       tokenizer="distilbert-base-uncased-distilled-squad")

context = """The Linux kernel project is a free and open-source, monolithic, Unix-like operating system kernel. 
The Linux kernel was conceived and created in 1991 by Linus Torvalds for his personal computer, and it has since 
grown to support a wide variety of computer architectures, including the x86, ARM, and RISC-V instruction sets. 
The Linux kernel is released under the GNU General Public License version 2 (GPLv2), which allows anyone to view, 
modify, and distribute the source code. The kernel is developed by a large community of developers from around 
the world, with contributions from individuals, companies, and organizations. It is used as the foundation for 
many operating systems, including popular distributions such as Ubuntu, Fedora, and Debian.
"""

questions = [
    "Who created the Linux kernel?",
    "When was the Linux kernel created?",
    "What is the license of the Linux kernel?",
    "What is the Linux kernel used for?",
    "What is the Linux kernel?",
    "Which are popular distributions that use the Linux kernel?"
]

for question in questions:
    answer = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {answer['answer']}\n")
    
    #inputs = tokenizer(question, context, return_tensors="pt")
    #outputs = model(**inputs)
    #answer_start_index = outputs.start_logits.argmax()
    #answer_end_index = outputs.end_logits.argmax()
    #answer = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][answer_start_index:answer_end_index+1]))
    #print(f"Q: {question}")
    #print(f"A: {answer}\n")


Device set to use mps:0


Q: Who created the Linux kernel?
A: Linus Torvalds

Q: When was the Linux kernel created?
A: 1991

Q: What is the license of the Linux kernel?
A: GNU General Public License version 2

Q: What is the Linux kernel used for?
A: the foundation for 
many operating systems

Q: What is the Linux kernel?
A: free and open-source, monolithic, Unix-like operating system kernel

Q: Which are popular distributions that use the Linux kernel?
A: Ubuntu, Fedora, and Debian



---

### Text Generation Example

In [83]:
# Text Generation example using DeepSeek-R1-Distill-Qwen-1.5B model
from transformers import pipeline
#from accelerate import Accelerator
#device = Accelerator().device

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
text_gen = pipeline("text-generation", model=model_name)

result = text_gen("The distance between the Earth and the Moon is approximately", 
                   temperature=1.2)

print(result[0]['generated_text'])

Device set to use mps:0


The distance between the Earth and the Moon is approximately \(384 \mathrm{~mm}\). Suppose that the Earth and the Moon are perfectly circular...
The question is: [1] Prove that the triangle... The distance is approximately \(384 \mathrm{~mm}\). Suppose that the Earth and the Moon are perfectly circular and perfectly aligned.
Now, [2] Prove that the...
Hmm, it's been a while since I last solved a geometry problem, so maybe I need to take things step by step.

First, I should try to visualize or sketch the setup because I'm not sure of the exact details. I know that the distance between the Earth and the Moon is approximately 384,000 km because typically, we see something like that, but wait, 384 mm is way smaller. So, 384 mm between them, okay.

So, in this problem, it's stated that both the Earth and the Moon are perfectly circular and perfectly aligned. So, that suggests that they're both in the same plane, and perhaps at an angle relative to each other so that when viewed from Earth,

In [ ]:
# Text Generation example without using pipeline()
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

model = AutoModelForCausalLM.from_pretrained(model_name")

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "The distance between the earth and the moon in kilometers is approximately"
model_inputs = tokenizer(text, return_tensors="pt").to(model.device)

generated_ids = model.generate(**model_inputs, max_new_tokens=500, temperature=0.2, do_sample=True)
result = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(result)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


The distance between the earth and the moon in kilometers is approximately 1.38 × 10^6 km. If the moon is moving in a circular orbit around the earth, what is the moon's speed in km/s? (Assume that the moon's orbit is circular and that the moon's orbital period is 1.38 × 10^6 s.)
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should the answer be rounded?
To which decimal place should 

In [84]:
# Text Generation example using Gemma-3-1B-IT model
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator

device = Accelerator().device

model_name = "google/gemma-3-1b-it"

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "The distance between the earth and the moon in kilometers is approximately"
model_inputs = tokenizer(text, return_tensors="pt").to(device)

generated_ids = model.generate(**model_inputs, temperature=0.6, do_sample=True)
result = tokenizer.decode(generated_ids[0])
print(result)

<bos>The distance between the earth and the moon in kilometers is approximately 384,400 kilometers.

The moon's orbit around the Earth is


In [85]:
# Basic Prompt Engineering Example

from transformers import pipeline

text_gen = pipeline("text-generation", model="google/gemma-3-1b-it")

prompt = """Classify each of the following texts into one of the following categories: Science, Philosophy, History, Geography.

Text: The distance between the Earth and the Moon is approximately 384,400 kilometers.
Text: The Zen principles emphasize meditation and intuition rather than ritual worship or study of scriptures.
Text: The Great Wall of China is a series of fortifications built along the northern borders of China.
Text: Delhi is the capital of India and is known for its rich history and cultural heritage.

"""

outputs = text_gen(prompt, temperature=0.7, max_new_tokens=200, do_sample=True, num_return_sequences=1)
for output in outputs:
    print(output['generated_text'])
    print("\n---\n")

Device set to use mps:0


Classify each of the following texts into one of the following categories: Science, Philosophy, History, Geography.

Text: The distance between the Earth and the Moon is approximately 384,400 kilometers.
Text: The Zen principles emphasize meditation and intuition rather than ritual worship or study of scriptures.
Text: The Great Wall of China is a series of fortifications built along the northern borders of China.
Text: Delhi is the capital of India and is known for its rich history and cultural heritage.

**Answer:**

1.  Science: The distance between the Earth and the Moon is approximately 384,400 kilometers.
2.  Philosophy: The Zen principles emphasize meditation and intuition rather than ritual worship or study of scriptures.
3.  History: The Great Wall of China is a series of fortifications built along the northern borders of China.
4.  Geography: Delhi is the capital of India and is known for its rich history and cultural heritage.

**Explanation:**

*   **Science:** This text de

In [86]:
from transformers import pipeline

pipe = pipeline("text-generation", 
                model="google/gemma-3-1b-it", 
                device="mps", dtype="auto")

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are William Shakespeare"},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on AI using Python Language"},]
        },
    ],
]

output = pipe(messages, max_new_tokens=500)
print(output)
print("-" * 30)
print(output[0][0]['generated_text'][-1]["content"])

Device set to use mps


[[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are William Shakespeare'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Write a poem on AI using Python Language'}]}, {'role': 'assistant', 'content': "Hark, gentle souls, and lend a listening ear,\nTo tales of metal, banishing all fear!\nA new creation, born of logic bright,\nA Python script, a wondrous, digital light.\n\nNo flesh and bone, no breath to warm the soul,\nYet learns and grows, beyond our own control.\n'Tis 'tis a language, coded deep and vast,\nWhere algorithms dance, and futures cast.\n\nWe feed it data, rivers flowing free,\nOf Shakespeare’s verse, of history’s decree.\nIt parses words, and understands their grace,\nAnd crafts new sonnets in this digital space.\n\nIt mimics thought, a clever, subtle art,\nTo answer queries, playing a vital part.\nA chatbot bright, with voices smooth and clear,\nDispelling doubt, and banishing all fear.\n\nIt paints with pixels, mimics human h

---
### Sentiment Analysis

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

classifier(["Python + ML skills = Super-human capabilities",
           "World Models are the future of AI",
           "Building neural networks using Assembly language can be painful"])

---

### Text Summarization example

In [87]:
# Text Summarization Example
# pipeline() does not work with "summarization" task on transformers 5.x.
# So, we need to initialize the model and tokenizer separately.

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from accelerate import Accelerator
device = Accelerator().device

model_name = "facebook/bart-large-cnn"

text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

summarizer = pipeline(task="summarization", 
                      model=model_name, 
                      tokenizer=model_name, 
                      device=device)

print("Summary:")
result = summarizer(text, max_length=150, min_length=40, do_sample=False)
print(result)
print("-" * 30)

summary = result[0]['summary_text']

print(summary.replace(". ", ".\n"))


Device set to use mps


Summary:
[{'summary_text': 'America has changed dramatically during recent years. The number of engineering graduates in the U.S. is on the decline. China and India graduate six and eight times as many traditional engineers as does the United States.'}]
------------------------------
America has changed dramatically during recent years.
The number of engineering graduates in the U.S.
is on the decline.
China and India graduate six and eight times as many traditional engineers as does the United States.


In [ ]:
# Text Summarization Example
# pipeline() does not work with "summarization" task on transformers 5.x.
# This code should work with transformers 5.x.
# So, we need to initialize the model and tokenizer separately.

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from accelerate import Accelerator
device = Accelerator().device

model_name = "facebook/bart-large-cnn"

text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(device)

generated_ids = model.generate(**inputs,  
                               num_beams=4, 
                               length_penalty=0.9, 
                               no_repeat_ngram_size=3, 
                               early_stopping=True)
summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Summary:")
print(summary.replace(". ", ".\n"))


Summary:
America has changed dramatically during recent years.
The number of engineering graduates in the U.S.
is on the decline.
China and India graduate six and eight times as many traditional engineers as the United States.
Other industrial countries maintain their output, while America struggles with a lack of well-educated engineers.


---

### Text to Speech synthesis example

In [88]:
# Text to Audio synthesis Example

from transformers import pipeline
from accelerate import Accelerator
device = Accelerator().device

#model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice" # Does not work with pipeline() on transformers 5.x
model_name = "suno/bark-small"
tts = pipeline("text-to-speech", model=model_name, device=device)

print("Generating audio from text...")

text = "This is a sample text to demonstrate the text-to-speech synthesis capabilities of the model."

audio = tts(text)
print(audio)

# The Audio can be played directly in a Jupyter Notebook as below:
from IPython.display import Audio
Audio(audio["audio"], rate=audio["sampling_rate"])

Device set to use mps


Generating audio from text...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{'audio': array([[ 5.2895927e-04, -1.5635043e-05,  1.6055908e-04, ...,
         1.1942168e-01,  1.1341388e-01,  1.0817204e-01]],
      shape=(1, 205760), dtype=float32), 'sampling_rate': 24000}


In [ ]:
from IPython.display import Audio
Audio(audio["audio"], rate=audio["sampling_rate"])

In [ ]:
# Suitable for transformers 5.x, using AutoProcessor and BarkModel directly
import torch
from transformers import AutoProcessor, BarkModel
from accelerate import Accelerator

device = Accelerator().device
model_name = "suno/bark-small"
# Load the processor and model explicitly
processor = AutoProcessor.from_pretrained(model_name)
model = BarkModel.from_pretrained(model_name).to(device)

text = "This is a sample text to demonstrate the text-to-speech synthesis capabilities of the model."

# 1. Manually prepare inputs with attention_mask
inputs = processor(text, return_tensors="pt").to(device)

print("Generating audio tokens...")

# 2. Feed attention_mask and define pad_token_id to eliminate the warning
audio_tokens = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    pad_token_id=processor.tokenizer.eos_token_id  # Force padding to map to EOS
)
# Convert generated audio tokens back to a numpy array for playback/saving
audio_array = audio_tokens.cpu().numpy().squeeze()

print("Audio generation complete.")

from IPython.display import Audio
Audio(audio_array, rate=16000)  # Assuming a sample rate of 16k

In [89]:
from transformers import pipeline

tts_pipeline = pipeline("text-to-speech", model="facebook/mms-tts-eng")

text_prompt = "The Hugging Face pipeline makes audio generation incredibly easy."
output = tts_pipeline(text_prompt)
Audio(output["audio"], rate=output["sampling_rate"])

Device set to use mps:0


---

### Automatic Speech Recognition (ASR) Example

In [91]:
pwd

'/Users/chandra/Training/HuggingFace_Transformers/Samples/Day_2'

In [92]:
# Automatic Speech Recognition (ASR) Example

from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="distil-whisper/distil-small.en")
asr_result = asr("../../harvard.wav")
asr_result["text"]

Device set to use mps:0
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


' The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.'

---

### Text to Image Generation Example

In [93]:
!conda install diffusers -c conda-forge -y

Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

# All requested packages already installed.

WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html



In [94]:
# Text to Image Generation Example
import torch
from diffusers import StableDiffusionPipeline
from accelerate import Accelerator
device = Accelerator().device

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16
)
pipe = pipe.to(device)

prompt = "a photorealistic image of an astronaut riding a horse in a futuristic city, digital art"
result = pipe(prompt)
print(result)
image = result.images[0]
image.save("../astronaut_riding_horse.png")
image.show()

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

StableDiffusionPipelineOutput(images=[<PIL.Image.Image image mode=RGB size=512x512 at 0x336365F50>], nsfw_content_detected=[False])


---

### Image Segmentation using ViT

In [ ]:
from transformers import pipeline
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/segmentation_input.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image

In [ ]:
from transformers import pipeline
semantic_segmentation = pipeline("image-segmentation", 
                                 "nvidia/segformer-b1-finetuned-cityscapes-1024-1024")

results = semantic_segmentation(image)
results

In [ ]:
print(results[9]["label"], results[9]["score"], results[9]["mask"])

---

## Encoder-Decoder use-cases

### Text Translation

In [20]:
!conda install -c conda-forge sentencepiece -y

Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

## Package Plan ##

  environment location: /opt/anaconda3/envs/hf4_env

  added / updated specs:
    - sentencepiece


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libsentencepiece-0.2.1     |       h9466b84_3         761 KB  conda-forge
    sentencepiece-0.2.1        |       hd55af37_3          20 KB  conda-forge
    sentencepiece-python-0.2.1 |  py311h40cb3f3_3         3.2 MB  conda-forge
    sentencepiece-spm-0.2.1    |       h9466b84_3          84 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         4.1 MB

The following NEW packages will be INSTALLED:

  libsentencepiece   conda-forge/osx-arm64::libsentencepiece-0.2.1-h9466b84_3 
  sentencepiece      conda-forge/osx-arm64::sentencepiece-0.2.1-hd55af37_3 
  sente

In [1]:
from transformers import pipeline

# Helsinki-NLP models for 1000+ language pairs
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr"
)
result = translator("Hello, how are you?")
# [{'translation_text': 'Bonjour, comment allez-vous?'}]

print(result[0]['translation_text'])


source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/opt/anaconda3/envs/hf4_env/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use mps:0


Bonjour, comment allez-vous ?


##### Language Code Formats:
- Helsinki-NLP: ISO 639-1 (`en`, `fr`, `de`, `zh`)
- NLLB: FLORES-200 codes (`eng_Latn`, `fra_Latn`, `zho_Hans`)
- M2M-100: Similar to FLORES codes


In [5]:

# Multilingual model (NLLB - No Language Left Behind)
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M",
    src_lang="eng_Latn",  # Source language code
    tgt_lang="hin_Deva"   # Target language code
)
result = translator("Hello, how are you?")
print(result[0]['translation_text'])

Device set to use mps:0


हैलो, आप कैसे हैं?


#### Summarization Example

In [6]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

article = """The transformer is a deep learning architecture introduced 
in the 2017 paper 'Attention Is All You Need' by Google researchers. 
Unlike previous models that processed sequences sequentially, the 
transformer uses self-attention mechanisms to process all tokens 
simultaneously, enabling much better parallelization. This breakthrough 
has led to significant advances in natural language processing, 
including models like BERT, GPT, and T5. The architecture has since 
been adapted for computer vision, speech recognition, and multimodal 
applications."""

summary = summarizer(
    article,
    max_length=50,
    min_length=20,
    do_sample=False
)
print(summary[0]['summary_text'])
# "The transformer architecture uses self-attention to process 
#  sequences in parallel. This has led to advances in NLP models 
#  like BERT, GPT, and T5."

Device set to use mps:0


The transformer is a deep learning architecture introduced by Google researchers in 2017. It uses self-attention mechanisms to process all tokens simultaneously.


---

### Text-to-Text Generation

In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")

# Grammar correction
input_text = "grammar: He have been working here since five years."
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_length=128)
corrected = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(corrected)
# "He has been working here for five years."

# Question generation from context
context = """The Eiffel Tower is a wrought-iron lattice tower on the 
Champ de Mars in Paris, France. It is named after the engineer 
Gustave Eiffel, whose company designed and built the tower."""
input_text = f"generate question: {context}"
# Output: "Who designed the Eiffel Tower?"
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_length=128)
question = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(question)


config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

he have been working here since five years.
True


---

## Hands-on Lab: Feature Extraction with BERT (15 mins)



---
### Exercise 1: Getting Contextual Embeddings


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

# Load model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Sample sentences
sentences = [
    "I love programming in Python.",
    "The snake python is a non-venomous constrictor.",
    "Python is both a programming language and a snake species."
]

# Tokenize
inputs = tokenizer(sentences, padding=True, truncation=True, 
                   return_tensors="pt")

# Get embeddings
with torch.no_grad():
    outputs = model(**inputs)

# Extract embeddings
last_hidden_states = outputs.last_hidden_state  # [3, seq_len, 768]
cls_embeddings = last_hidden_states[:, 0, :]     # [3, 768]

# Show "Python" token embedding differs based on context!
for i, sent in enumerate(sentences):
    tokens = tokenizer.tokenize(sent)
    if 'python' in tokens:
        idx = tokens.index('python') + 1  # +1 for [CLS]
        print(f"Sentence: {sent}")
        print(f"  python embedding shape: {last_hidden_states[i, idx].shape}")
        print(f"  First 5 values: {last_hidden_states[i, idx, :5].tolist()}\n")


---

### Lab Exercise 2: Cosine Similarity of Embeddings


In [ ]:
import torch.nn.functional as F

# Get word embeddings for "python" in different contexts
def get_word_embedding(model, tokenizer, sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    tokens = tokenizer.tokenize(sentence)
    word_tokens = tokenizer.tokenize(word)
    
    # Find position (handle subword tokens)
    for i in range(len(tokens) - len(word_tokens) + 1):
        if tokens[i:i+len(word_tokens)] == word_tokens:
            # Average embeddings if word is split
            return outputs.last_hidden_state[0, i+1:i+1+len(word_tokens)].mean(0)
    return None

# Compare "python" across sentences
emb1 = get_word_embedding(model, tokenizer, 
                          "I love programming in Python.", "python")
emb2 = get_word_embedding(model, tokenizer, 
                          "The python snake is dangerous.", "python")
emb3 = get_word_embedding(model, tokenizer, 
                          "I code in Python.", "python")

# Compute similarities
sim_12 = F.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))
sim_13 = F.cosine_similarity(emb1.unsqueeze(0), emb3.unsqueeze(0))
sim_23 = F.cosine_similarity(emb2.unsqueeze(0), emb3.unsqueeze(0))

print(f"Similarity: Programming Python vs Snake Python: {sim_12.item():.3f}")
print(f"Similarity: Programming Python vs Code Python: {sim_13.item():.3f}")
print(f"Similarity: Snake Python vs Code Python: {sim_23.item():.3f}")
print("\nNote: Context matters! Same word, different embeddings.")



---

### Lab Exercise 3: Text Generation with GPT-2


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load GPT-2
model_name = "gpt2"  # 124M parameters - runs on CPU!
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Add padding token (GPT-2 doesn't have one by default)
tokenizer.pad_token = tokenizer.eos_token

# Test different generation strategies
prompt = "In a distant future, artificial intelligence"

strategies = {
    "Greedy": {
        "do_sample": False, "max_length": 60, "num_beams": 1
    },
    "Beam Search (k=5)": {
        "do_sample": False, "max_length": 60, "num_beams": 5,
        "early_stopping": True
    },
    "Sampling (temp=0.7)": {
        "do_sample": True, "max_length": 60, "temperature": 0.7,
        "top_k": 0
    },
    "Sampling (temp=1.0, top_p=0.9)": {
        "do_sample": True, "max_length": 60, "temperature": 1.0,
        "top_p": 0.9, "top_k": 50
    }
}

for strategy, params in strategies.items():
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, **params)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n{'='*50}")
    print(f"{strategy}:")
    print(f"{'='*50}")
    print(generated)
    print()



---

### Lab Exercise 4: Comparing Generation Parameters


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

prompt = "Once upon a time"

# Generate with different temperatures
temperatures = [0.1, 0.3, 0.5, 0.7, 0.9, 1.2, 1.5]

print("Effect of Temperature on Generation:\n")
for temp in temperatures:
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_length=50,
        do_sample=True,
        temperature=temp,
        top_k=50,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Temperature {temp}: {generated}\n")

# Observe:
# - Low temp: Repetitive, deterministic
# - Medium temp: Coherent, natural
# - High temp: Creative but potentially incoherent



---

### Lab Exercise 5: Comparing Model Sizes


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

# Compare different model sizes (if available memory)
models_to_test = [
    ("gpt2", "GPT-2 (124M)"),
    ("distilgpt2", "DistilGPT-2 (82M)"),
]

for model_name, display_name in models_to_test:
    try:
        print(f"\nLoading {display_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)
        
        prompt = "The future of AI is"
        inputs = tokenizer(prompt, return_tensors="pt")
        
        # Time the generation
        start = time.time()
        outputs = model.generate(
            **inputs,
            max_length=50,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
        elapsed = time.time() - start
        
        params = sum(p.numel() for p in model.parameters())
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        print(f"Parameters: {params:,}")
        print(f"Generation time: {elapsed:.2f}s")
        print(f"Generated: {generated[:100]}...")
        
        # Clean up memory
        del model, tokenizer
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
        
    except Exception as e:
        print(f"  Failed: {e}")



---

# Hands-on Lab: Translation and Summarization (15 mins)

---

## Exercise 1: Multi-Language Translation


In [ ]:
from transformers import pipeline
import torch

# Load translation pipeline
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-en-fr",
    device=0 if torch.cuda.is_available() else -1
)

# Translate sentences
english_texts = [
    "The weather is beautiful today.",
    "Machine learning is transforming industries.",
    "I would like a cup of coffee, please."
]

for text in english_texts:
    result = translator(text)
    print(f"EN: {text}")
    print(f"FR: {result[0]['translation_text']}\n")

# Try other language pairs
translators = {
    "en→fr": "Helsinki-NLP/opus-mt-en-fr",
    "en→de": "Helsinki-NLP/opus-mt-en-de",
    "en→es": "Helsinki-NLP/opus-mt-en-es",
    "en→it": "Helsinki-NLP/opus-mt-en-it",
    "en→nl": "Helsinki-NLP/opus-mt-en-nl",
}

text = "Hello, how are you?"
for lang_pair, model_name in translators.items():
    t = pipeline("translation", model=model_name)
    result = t(text)
    print(f"{lang_pair}: {result[0]['translation_text']}")



---

### Lab Exercise 2: Summarization with BART


In [ ]:
from transformers import pipeline

# Load summarization pipeline
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1  # Use CPU if GPU memory limited
)

# Sample long text
long_text = """
Artificial intelligence has made remarkable progress in recent years, 
particularly in the field of natural language processing. Large language 
models like GPT-3, BERT, and their successors have demonstrated impressive 
capabilities in understanding and generating human-like text. These models 
are trained on massive datasets containing billions of words from the 
internet, books, and other sources. Through a process called pre-training, 
they learn patterns in language, grammar, and even some reasoning abilities. 

However, these models also have significant limitations. They can generate 
plausible-sounding but factually incorrect information, exhibit biases 
present in their training data, and require enormous computational resources 
to train and deploy. Researchers are actively working on making these models 
more efficient, accurate, and aligned with human values. The field continues 
to evolve rapidly, with new architectures and training methods being 
developed regularly.
"""

# Generate summaries with different lengths
for length_ratio in [0.3, 0.5, 0.7]:
    # Estimate max_length based on input
    input_length = len(long_text.split())
    max_len = int(input_length * length_ratio)
    min_len = max_len // 2
    
    summary = summarizer(
        long_text,
        max_length=max_len,
        min_length=min_len,
        do_sample=False
    )
    print(f"\nSummary ({length_ratio:.0%} length, ~{max_len} words):")
    print(summary[0]['summary_text'])


---

### Lab Exercise 3: Image Classification with ViT


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# Load image classification pipeline
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

# Load a sample image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

# Display image
plt.imshow(image)
plt.axis('off')
plt.show()

# Classify
results = classifier(image)
for result in results[:5]:
    print(f"{result['label']}: {result['score']:.3f}")

# Try with your own images
# image = Image.open("path/to/your/image.jpg")
# results = classifier(image)



---

### Lab Exercise 4: Zero-Shot with CLIP


In [ ]:
from transformers import pipeline

# Load zero-shot image classification pipeline
classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

# Load image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
import requests
from PIL import Image
from io import BytesIO

response = requests.get(url)
image = Image.open(BytesIO(response.content))

# Define candidate labels (not limited to training labels!)
candidate_labels = [
    "a cat",
    "a dog",
    "a tiger",
    "a lion",
    "a rabbit",
    "a car",
    "a building",
    "food"
]

# Classify
results = classifier(image, candidate_labels=candidate_labels)
for result in results:
    print(f"{result['label']}: {result['score']:.3f}")

# Try with creative/abstract labels
creative_labels = [
    "a majestic feline",
    "a internet meme star",
    "a sleepy animal",
    "a predator",
    "someone's pet"
]

results = classifier(image, candidate_labels=creative_labels)
print("\nCreative labeling:")
for result in results:
    print(f"{result['label']}: {result['score']:.3f}")


---

### Lab Exercise 5: Speech Recognition with Whisper


In [ ]:
from transformers import pipeline
import torch

# Load ASR pipeline (use tiny model for quick testing)
transcriber = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-tiny",
    device=0 if torch.cuda.is_available() else -1
)

# If you have an audio file:
# audio_file = "path/to/audio.mp3"
# result = transcriber(audio_file)
# print(f"Transcription: {result['text']}")

# For testing without audio file - create a note about setup
print("To test speech recognition:")
print("1. Install required packages:")
print("   pip install librosa soundfile")
print()
print("2. Use with audio file:")
print('   result = transcriber("audio.mp3")')
print('   print(result["text"])')
print()
print("Whisper models available:")
print("  whisper-tiny:   39M parameters (fastest)")
print("  whisper-base:   74M parameters")
print("  whisper-small:  244M parameters")
print("  whisper-medium: 769M parameters")
print("  whisper-large:  1.5B parameters (best quality)")

# Demo with a sample from HuggingFace dataset
from datasets import load_dataset

print("\nLoading sample from LibriSpeech...")
try:
    # Load a sample
    dataset = load_dataset("librispeech_asr", "clean", split="test", streaming=True)
    sample = next(iter(dataset))
    
    # Transcribe
    result = transcriber(sample["audio"]["array"])
    print(f"Transcription: {result['text']}")
    print(f"Reference: {sample['text']}")
except Exception as e:
    print(f"Couldn't load sample: {e}")
    print("This is expected if you don't have audio dependencies installed.")


---

### Lab Exercise 6: Multi-Model Pipeline Chain


In [ ]:
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Create a chain: Image → Caption → Translation
print("Creating multi-model pipeline chain...")

# 1. Image captioning
captioner = pipeline(
    "image-to-text",
    model="nlpconnect/vit-gpt2-image-captioning"
)

# 2. Translation (English → French)
translator = pipeline(
    "translation_en_to_fr",
    model="Helsinki-NLP/opus-mt-en-fr"
)

# 3. Sentiment analysis on caption
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Process an image through the chain
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
response = requests.get(url)
image = Image.open(BytesIO(response.content))

print("\n1. Generating caption...")
caption = captioner(image)[0]['generated_text']
print(f"   Caption: {caption}")

print("\n2. Translating to French...")
translated = translator(caption)[0]['translation_text']
print(f"   French: {translated}")

print("\n3. Analyzing sentiment of caption...")
sentiment = sentiment_analyzer(caption)[0]
print(f"   Sentiment: {sentiment['label']} (confidence: {sentiment['score']:.3f})")

print("\n" + "="*50)
print("Pipeline Chain Complete!")
print(f"  Input: Image of a cat")
print(f"  → Caption: {caption}")
print(f"  → Translation: {translated}")
print(f"  → Sentiment: {sentiment['label']}")

